**Planning in CrewAI means creating a step-by-step plan for the crew’s tasks before the agents carry them out.**

**PLANNING WITH THE DEFAULT PLANNING MODEL - When you set planning=True, CrewAI uses an AgentPlanner to prepare guidance and adds that plan to each task description.**

In [6]:
from crewai import Agent, Task, Crew

agent = Agent(
    role="AI Teacher",
    goal="Explain AI concepts simply",
    backstory="You teach beginners."
)

task = Task(
    description="Explain three everyday uses of Artificial Intelligence.",
    expected_output="Three short, beginner-friendly examples.",
    agent=agent
)

crew = Crew(
    agents=[agent],
    tasks=[task],
    planning=True
)

result = await crew.kickoff_async()
print(result)

1. Voice assistants like Siri or Alexa listen to your questions and help by giving answers or doing tasks, making it easier to use your phone or smart devices.  
2. Recommendation systems on websites like Netflix or YouTube suggest movies or videos you might enjoy, saving you time by finding things you like.  
3. Navigation apps like Google Maps show the best route to your destination, helping you avoid traffic and get there faster.


**PLANNING WITH A CHOOSEN planning_llm**

In [7]:
from crewai import Agent, Task, Crew

agent = Agent(
    role="AI Teacher",
    goal="Explain AI concepts simply",
    backstory="You teach beginners.",
)

task = Task(
    description="Explain three everyday uses of Artificial Intelligence.",
    expected_output="Three short, beginner-friendly examples.",
    agent=agent
)

crew = Crew(
    agents=[agent],
    tasks=[task],
    planning=True,
    planning_llm="gpt-4o"
)

result = await crew.kickoff_async()
print(result)

Artificial Intelligence, or AI, is a technology that allows machines to think and act like humans. It helps computers perform tasks that usually need human intelligence, such as understanding language, learning from experience, and making decisions.

Here are three everyday ways AI is used:

1. **Personal Assistants (like Siri and Alexa):**  
AI powers virtual helpers on your phone or smart speakers. When you speak to them, they understand your requests and can do things like set reminders, answer questions, or control lights and music in your home. Over time, they learn what you like, making them more helpful and saving you time.

2. **Recommendation Systems (like Netflix and Amazon):**  
AI looks at what you have watched or bought before and suggests movies, shows, or products you might enjoy next. By learning from your past choices, it helps you find things you want faster, making your experience more personal and fun.

3. **Autonomous Vehicles (self-driving cars):**  
AI helps cars

**TESTING**

**Testing in CrewAI means running a crew multiple times and evaluating how well its agents complete their tasks. CrewAI’s built-in testing shows scores for each task and the crew overall, helping you judge how consistent the results are.**

In [1]:
import os

os.environ["CREWAI_DISABLE_TELEMETRY"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"

In [5]:
import os
import subprocess
import sys

from dotenv import load_dotenv

load_dotenv()

test_code = """
from crewai import Agent, Task, Crew, LLM

teacher = Agent(
    role="AI Teacher",
    goal="Explain AI concepts simply",
    backstory="You teach beginners with clear examples.",
    llm=LLM(model="gpt-4o-mini", temperature=0),
)

task = Task(
    description="Explain {topic} in three simple sentences.",
    expected_output="Three beginner-friendly sentences.",
    agent=teacher,
)

crew = Crew(agents=[teacher], tasks=[task])

crew.test(
    n_iterations=2,
    eval_llm="gpt-4o-mini",
    inputs={"topic": "Artificial Intelligence"},
)
"""

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Set OPENAI_API_KEY in your .env file.")

run = subprocess.run(
    [sys.executable, "-c", test_code],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    env={**os.environ, "PYTHONIOENCODING": "utf-8"},
)

for line in (run.stdout + run.stderr).splitlines():
    if "on_crew_test_result" not in line and "no attribute 'crew'" not in line:
        print(line)

if run.returncode != 0:
    raise RuntimeError(f"CrewAI test exited with code {run.returncode}")

result = await crew.kickoff_async(
    inputs={"topic": "Artificial Intelligence"}
)
print("\nAnswer:\n", result.raw)



                           Tasks Scores                           
                      (1-10 Higher is better)                     
┌────────────────────┬───────┬───────┬────────────┬──────────────┐
│ Tasks/Crew/Agents  │ Run 1 │ Run 2 │ Avg. Total │ Agents       │
├────────────────────┼───────┼───────┼────────────┼──────────────┤
│ Task 1             │  8.0  │  8.0  │    8.0     │ - AI Teacher │
│ Crew               │ 8.00  │ 8.00  │    8.0     │              │
│ Execution Time (s) │   3   │   1   │     2      │              │
└────────────────────┴───────┴───────┴────────────┴──────────────┘

Answer:
 Artificial Intelligence, or AI, is a branch of computer science that aims to create machines that can think and learn like humans. It uses algorithms and data to perform tasks such as recognizing speech, making decisions, and solving problems. Essentially, AI helps computers understand and respond to the world around them, making them more useful in everyday life.
